# Mislabel Detection via TF-IDF + Out-of-Fold Probabilities

## Approach

Toxicity annotation is subjective. Crowd-sourced NLP label noise studies typically find **3–8%** mislabeling rates. We detect suspected mislabels by:

1. **TF-IDF Features** — word unigrams/bigrams (100k features)
2. **5-Fold Cross-Validation** — train Logistic Regression on 4 folds, collect predicted probabilities on the held-out fold for every sample
3. **Out-of-Fold (OOF) Probabilities** — each sample gets a probability from a model that never trained on it → unbiased signal
4. **Flag outliers** — samples where the model is highly confident but disagrees with the given label:
   - **Suspected false negatives**: `label=0` but `p_oof ≥ 0.80` (model thinks it's toxic)
   - **Suspected false positives**: `label=1` but `p_oof ≤ 0.20` (model thinks it's clean)

### Why these thresholds?
- A random disagreement at p=0.51 means nothing — the model is uncertain
- At p≥0.80 the model is *highly confident*, so the label is the more likely culprit
- We also report a softer threshold (p≥0.65) for a broader suspect list

### Expected ballpark (before running)
| Category | Rough estimate |
|---|---|
| Toxic labeled non-toxic (p≥0.80) | ~4,000–7,000 (~2.7–4.7% of non-toxic class) |
| Non-toxic labeled toxic (p≤0.20) | ~450–750 (~3–5% of toxic class) |
| **Total high-confidence suspects** | **~3–5% of 159k training samples** |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('../data')
LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Thresholds for flagging mislabels
HIGH_CONF = 0.80   # very likely mislabeled
MED_CONF  = 0.65   # possibly mislabeled (broader suspect list)
N_FOLDS   = 5
RANDOM_STATE = 42

## 1. Load Data

In [ ]:
df = pd.read_csv(DATA_DIR / 'train.csv')
print(f'Shape: {df.shape}')
print('\nLabel distribution:')
label_counts = df[LABELS].sum()
label_pcts   = df[LABELS].mean() * 100
display(pd.DataFrame({'count': label_counts, 'pct': label_pcts.round(2)}))

## 2. TF-IDF Features

Word unigrams + bigrams, 100k features, `sublinear_tf=True`.

In [ ]:
texts = df['comment_text'].fillna('').values

print('Fitting word TF-IDF...')
word_tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=100_000,
    sublinear_tf=True,
    strip_accents='unicode',
    token_pattern=r'\w{2,}',
    min_df=3,
)
X = word_tfidf.fit_transform(texts)
print(f'Feature matrix shape: {X.shape}')

## 3. Cross-Validated Out-of-Fold Probabilities

For each label independently, run 5-fold CV and store the held-out probability. This is the cornerstone of the approach — every sample gets a probability from a model that never saw it during training.

In [ ]:
oof_probs = np.zeros((len(df), len(LABELS)))
auc_scores = {}

for label_idx, label in enumerate(LABELS):
    y = df[label].values
    oof = np.zeros(len(y))

    # Stratify on binary label to preserve class ratio in each fold
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    fold_aucs = []
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        clf = LogisticRegression(
            C=0.3,
            solver='lbfgs',
            max_iter=1000,
            class_weight='balanced',
        )
        clf.fit(X[train_idx], y[train_idx])
        oof[val_idx] = clf.predict_proba(X[val_idx])[:, 1]

        fold_auc = roc_auc_score(y[val_idx], oof[val_idx])
        fold_aucs.append(fold_auc)

    oof_probs[:, label_idx] = oof
    mean_auc = np.mean(fold_aucs)
    auc_scores[label] = mean_auc
    print(f'{label:20s} OOF AUC = {mean_auc:.4f}  (folds: {[f"{a:.3f}" for a in fold_aucs]})')

print(f'\nMacro-avg OOF AUC: {np.mean(list(auc_scores.values())):.4f}')

In [ ]:
# Store OOF probs in df for easy analysis
for i, label in enumerate(LABELS):
    df[f'p_oof_{label}'] = oof_probs[:, i]

## 4. Flag Suspected Mislabels

For each label:
- **Suspected false negatives (FN)**: `label=0` but `p_oof ≥ HIGH_CONF` — model strongly predicts toxic, label says no
- **Suspected false positives (FP)**: `label=1` but `p_oof ≤ 1 - HIGH_CONF` — model strongly predicts clean, label says toxic

In [ ]:
summary_rows = []

for label in LABELS:
    y     = df[label].values
    p_oof = df[f'p_oof_{label}'].values

    n_pos = y.sum()
    n_neg = len(y) - n_pos

    # High-confidence suspects
    fn_mask_hi = (y == 0) & (p_oof >= HIGH_CONF)
    fp_mask_hi = (y == 1) & (p_oof <= 1 - HIGH_CONF)

    # Medium-confidence suspects (broader)
    fn_mask_md = (y == 0) & (p_oof >= MED_CONF)
    fp_mask_md = (y == 1) & (p_oof <= 1 - MED_CONF)

    summary_rows.append({
        'label':            label,
        'n_positive':       int(n_pos),
        'n_negative':       int(n_neg),
        # high confidence
        'fn_hi':            int(fn_mask_hi.sum()),
        'fn_hi_pct_neg':    round(fn_mask_hi.sum() / n_neg * 100, 2),
        'fp_hi':            int(fp_mask_hi.sum()),
        'fp_hi_pct_pos':    round(fp_mask_hi.sum() / n_pos * 100, 2),
        'total_hi':         int(fn_mask_hi.sum() + fp_mask_hi.sum()),
        'total_hi_pct_all': round((fn_mask_hi.sum() + fp_mask_hi.sum()) / len(y) * 100, 2),
        # medium confidence
        'fn_md':            int(fn_mask_md.sum()),
        'fp_md':            int(fp_mask_md.sum()),
        'total_md':         int(fn_mask_md.sum() + fp_mask_md.sum()),
        'total_md_pct_all': round((fn_mask_md.sum() + fp_mask_md.sum()) / len(y) * 100, 2),
    })

summary = pd.DataFrame(summary_rows).set_index('label')
display(summary)

In [ ]:
# Overall across all labels (a sample is flagged if any label is suspected)
all_fn_hi = np.zeros(len(df), dtype=bool)
all_fp_hi = np.zeros(len(df), dtype=bool)

for label in LABELS:
    y     = df[label].values
    p_oof = df[f'p_oof_{label}'].values
    all_fn_hi |= (y == 0) & (p_oof >= HIGH_CONF)
    all_fp_hi |= (y == 1) & (p_oof <= 1 - HIGH_CONF)

all_suspects_hi = all_fn_hi | all_fp_hi

print('=== HIGH-CONFIDENCE MISLABEL SUMMARY (threshold = {:.0%}) ==='.format(HIGH_CONF))
print(f'  Suspected false negatives (labeled 0, model says toxic): {all_fn_hi.sum():,}  ({all_fn_hi.mean()*100:.2f}% of all samples)')
print(f'  Suspected false positives (labeled 1, model says clean): {all_fp_hi.sum():,}  ({all_fp_hi.mean()*100:.2f}% of all samples)')
print(f'  Total unique flagged samples:                            {all_suspects_hi.sum():,}  ({all_suspects_hi.mean()*100:.2f}% of {len(df):,} training samples)')

## 5. Visualizations

### 5a. OOF Probability Distributions by True Label

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, label in zip(axes, LABELS):
    y     = df[label].values
    p_oof = df[f'p_oof_{label}'].values

    ax.hist(p_oof[y == 0], bins=80, alpha=0.55, color='steelblue',  label='label=0 (clean)', density=True)
    ax.hist(p_oof[y == 1], bins=80, alpha=0.55, color='tomato',     label='label=1 (toxic)', density=True)

    # Shade mislabel suspect zones
    ax.axvspan(HIGH_CONF, 1.0, alpha=0.12, color='tomato',   label=f'suspect FN (p≥{HIGH_CONF})')
    ax.axvspan(0.0, 1-HIGH_CONF, alpha=0.12, color='steelblue', label=f'suspect FP (p≤{1-HIGH_CONF})')

    ax.axvline(HIGH_CONF, color='red',  linestyle='--', linewidth=1.2)
    ax.axvline(1-HIGH_CONF, color='blue', linestyle='--', linewidth=1.2)

    fn_count = ((y == 0) & (p_oof >= HIGH_CONF)).sum()
    fp_count = ((y == 1) & (p_oof <= 1-HIGH_CONF)).sum()
    ax.set_title(f'{label}\nFN suspects: {fn_count:,}  |  FP suspects: {fp_count:,}', fontsize=10)
    ax.set_xlabel('OOF predicted probability')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)

plt.suptitle('OOF Probability Distributions — Mislabel Suspect Zones Highlighted', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results_mislabel_distributions.png', bbox_inches='tight')
plt.show()

### 5b. Mislabel Counts by Label and Threshold

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: absolute counts
x = np.arange(len(LABELS))
w = 0.35
axes[0].bar(x - w/2, summary['fn_hi'], w, label=f'FN suspects (p≥{HIGH_CONF})', color='tomato',    alpha=0.8)
axes[0].bar(x + w/2, summary['fp_hi'], w, label=f'FP suspects (p≤{1-HIGH_CONF})', color='steelblue', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(LABELS, rotation=30, ha='right')
axes[0].set_ylabel('Number of samples')
axes[0].set_title('High-confidence mislabel suspects by label')
axes[0].legend()

# Right: percentage of the respective class
axes[1].bar(x - w/2, summary['fn_hi_pct_neg'], w, label='FN% of negatives', color='tomato',    alpha=0.8)
axes[1].bar(x + w/2, summary['fp_hi_pct_pos'], w, label='FP% of positives', color='steelblue', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(LABELS, rotation=30, ha='right')
axes[1].set_ylabel('% of respective class')
axes[1].set_title('Mislabel rate as % of class')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results_mislabel_counts.png', bbox_inches='tight')
plt.show()

### 5c. Confidence Score Distribution of Suspects

In [ ]:
# Focus on the 'toxic' label (most important, most samples)
label = 'toxic'
y     = df[label].values
p_oof = df[f'p_oof_{label}'].values

fn_suspects = df[(y == 0) & (p_oof >= MED_CONF)].copy()
fn_suspects['p_oof'] = p_oof[(y == 0) & (p_oof >= MED_CONF)]
fn_suspects = fn_suspects.sort_values('p_oof', ascending=False)

fp_suspects = df[(y == 1) & (p_oof <= 1 - MED_CONF)].copy()
fp_suspects['p_oof'] = p_oof[(y == 1) & (p_oof <= 1 - MED_CONF)]
fp_suspects = fp_suspects.sort_values('p_oof')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(fn_suspects['p_oof'], bins=40, color='tomato', alpha=0.8, edgecolor='white')
axes[0].axvline(HIGH_CONF, color='darkred', linestyle='--', label=f'HIGH_CONF={HIGH_CONF}')
axes[0].set_xlabel('OOF probability (model says toxic)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'[toxic] FN suspects: {len(fn_suspects):,} (label=0, model says toxic)')
axes[0].legend()

axes[1].hist(fp_suspects['p_oof'], bins=40, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(1 - HIGH_CONF, color='navy', linestyle='--', label=f'LOW_CONF={1-HIGH_CONF}')
axes[1].set_xlabel('OOF probability (model says clean)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'[toxic] FP suspects: {len(fp_suspects):,} (label=1, model says clean)')
axes[1].legend()

plt.suptitle('"toxic" label — confidence distribution of suspects', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Inspect Examples

### 6a. Suspected False Negatives — toxic but labeled 0

In [ ]:
label = 'toxic'
y     = df[label].values
p_oof = df[f'p_oof_{label}'].values

fn_hi = df[(y == 0) & (p_oof >= HIGH_CONF)].copy()
fn_hi['p_oof'] = p_oof[(y == 0) & (p_oof >= HIGH_CONF)]
fn_hi = fn_hi.sort_values('p_oof', ascending=False)

print(f'Top 10 suspected false negatives for "{label}" (labeled 0, model very confident it is toxic):')
for _, row in fn_hi.head(10).iterrows():
    print(f'\n  p_oof={row["p_oof"]:.3f} | id={row["id"]}')
    print(f'  ["{row["comment_text"][:300]}"...' if len(row['comment_text']) > 300 else f'  "{row["comment_text"]}"')

### 6b. Suspected False Positives — labeled toxic but model says clean

In [ ]:
fp_hi = df[(y == 1) & (p_oof <= 1 - HIGH_CONF)].copy()
fp_hi['p_oof'] = p_oof[(y == 1) & (p_oof <= 1 - HIGH_CONF)]
fp_hi = fp_hi.sort_values('p_oof')

print(f'Top 10 suspected false positives for "{label}" (labeled 1, model very confident it is clean):')
for _, row in fp_hi.head(10).iterrows():
    print(f'\n  p_oof={row["p_oof"]:.3f} | id={row["id"]}')
    print(f'  ["{row["comment_text"][:300]}"...' if len(row['comment_text']) > 300 else f'  "{row["comment_text"]}"')

## 7. Build Cleaned Dataset

Two strategies are available. Choose based on your use case:

| Strategy | What it does | When to use |
|---|---|---|
| **Drop** | Remove high-confidence suspects entirely | When you'd rather have less, cleaner data |
| **Relabel** | Flip label to match the OOF prediction | When you want to preserve dataset size |

In [ ]:
STRATEGY = 'drop'   # change to 'relabel' to flip labels instead
THRESHOLD = HIGH_CONF  # use HIGH_CONF for conservative, MED_CONF for broader cleaning

df_clean = df[LABELS + ['id', 'comment_text']].copy()
flagged = np.zeros(len(df), dtype=bool)

for label in LABELS:
    y     = df[label].values
    p_oof = df[f'p_oof_{label}'].values

    fn_mask = (y == 0) & (p_oof >= THRESHOLD)
    fp_mask = (y == 1) & (p_oof <= 1 - THRESHOLD)

    if STRATEGY == 'relabel':
        df_clean.loc[fn_mask, label] = 1
        df_clean.loc[fp_mask, label] = 0
        print(f'{label}: relabeled {fn_mask.sum():,} FN + {fp_mask.sum():,} FP')
    else:
        flagged |= fn_mask | fp_mask

if STRATEGY == 'drop':
    df_clean = df_clean[~flagged].reset_index(drop=True)
    print(f'Dropped {flagged.sum():,} samples ({flagged.mean()*100:.2f}%).')
    print(f'Cleaned dataset size: {len(df_clean):,}  (was {len(df):,})')

print('\nNew label distribution:')
display(pd.DataFrame({'count': df_clean[LABELS].sum(), 'pct': (df_clean[LABELS].mean()*100).round(2)}))

In [ ]:
out_path = DATA_DIR / 'train_cleaned.csv'
df_clean.to_csv(out_path, index=False)
print(f'Saved cleaned training data to {out_path}')

## 8. Sanity Check — Does Cleaning Help?

Retrain on cleaned data with a simple 80/20 hold-out and compare AUC to the full noisy dataset.

In [ ]:
from sklearn.model_selection import train_test_split

def quick_auc(X_train, y_train, X_val, y_val):
    clf = LogisticRegression(C=0.3, solver='lbfgs', max_iter=1000, class_weight='balanced')
    clf.fit(X_train, y_train)
    preds = clf.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

label = 'toxic'
idx_clean = df_clean.index if STRATEGY == 'relabel' else df_clean.index

# Use original indices for alignment
train_idx, val_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE,
                                      stratify=df[label].values)

# Baseline (noisy)
auc_noisy = quick_auc(X[train_idx], df[label].values[train_idx],
                      X[val_idx],   df[label].values[val_idx])

# Cleaned — only train on non-flagged samples
if STRATEGY == 'drop':
    clean_train_mask = train_idx[~flagged[train_idx]]
    y_clean_train = df[label].values[clean_train_mask]
    auc_clean = quick_auc(X[clean_train_mask], y_clean_train,
                          X[val_idx],          df[label].values[val_idx])
else:
    y_clean_train = df_clean[label].values[train_idx]
    auc_clean = quick_auc(X[train_idx], y_clean_train,
                          X[val_idx],   df[label].values[val_idx])

print(f'[{label}]  Noisy AUC = {auc_noisy:.4f}  |  Cleaned AUC = {auc_clean:.4f}  |  Delta = {auc_clean - auc_noisy:+.4f}')

## 9. Summary

This cell prints a final human-readable report.

In [ ]:
print('=' * 65)
print('MISLABEL DETECTION REPORT')
print(f'Method:    TF-IDF (word 1-2g) + LogReg, {N_FOLDS}-fold CV')
print(f'Threshold: p ≥ {HIGH_CONF} (high-confidence) / p ≥ {MED_CONF} (medium-confidence)')
print(f'Dataset:   {len(df):,} training samples')
print('=' * 65)
print()
print(f'{"Label":<18} {"FN suspects":>12} {"FN%neg":>8} {"FP suspects":>12} {"FP%pos":>8} {"Total":>8} {"Total%":>7}')
print('-' * 75)
total_fn = total_fp = total_all = 0
for label in LABELS:
    row = summary.loc[label]
    print(f'{label:<18} {row["fn_hi"]:>12,} {row["fn_hi_pct_neg"]:>7.2f}% {row["fp_hi"]:>12,} {row["fp_hi_pct_pos"]:>7.2f}% {row["total_hi"]:>8,} {row["total_hi_pct_all"]:>6.2f}%')
    total_fn  += row['fn_hi']
    total_fp  += row['fp_hi']
    total_all += row['total_hi']
print('-' * 75)
print(f'{"(per-label totals)":<18} {total_fn:>12,} {"":>8} {total_fp:>12,}')
print()
print(f'Unique samples flagged (any label): {all_suspects_hi.sum():,}  ({all_suspects_hi.mean()*100:.2f}% of dataset)')
print()
print('Notes:')
print('  - "FN suspects" = labeled 0 (clean) but model p >= threshold  →  possibly missed toxic')
print('  - "FP suspects" = labeled 1 (toxic) but model p <= 1-threshold →  possibly mislabeled toxic')
print('  - Per-label totals can overlap (a sample may be suspect in multiple labels)')
print('  - Inspect examples in section 6 before deciding to drop/relabel')
print('=' * 65)

## 10. Test Set Mislabel Detection

The test set has **153,164** rows but only **63,978** carry real labels (the rest are `-1`, withheld from Kaggle scoring). We can still check those 63k for mislabels.

**Key difference from training:** we can't do OOF on test data — there's no held-out fold. Instead we train on the **full training set** and predict on the labeled test rows. Because the model never saw test data, the disagreement signal is still unbiased (just slightly less so than OOF, since the model used all training samples rather than 80%).

In [ ]:
# Load and merge test data — keep only rows with real labels (not -1)
test_text   = pd.read_parquet(DATA_DIR / 'test.parquet')
test_labels = pd.read_parquet(DATA_DIR / 'test_labels.parquet')

df_test = test_text.merge(test_labels, on='id')
df_test = df_test[df_test['toxic'] != -1].reset_index(drop=True)

print(f'Labeled test rows: {len(df_test):,}  (of {len(test_text):,} total)')
print('\nLabel distribution:')
display(pd.DataFrame({
    'count': df_test[LABELS].sum(),
    'pct':   (df_test[LABELS].mean() * 100).round(2)
}))

In [ ]:
# Transform test text with the already-fitted word TF-IDF (no refit — avoids data leakage)
X_test = word_tfidf.transform(df_test['comment_text'].fillna('').values)
print(f'Test feature matrix: {X_test.shape}')

In [ ]:
# Train one model per label on the full training set, predict on labeled test rows
test_probs = np.zeros((len(df_test), len(LABELS)))

for label_idx, label in enumerate(LABELS):
    clf = LogisticRegression(
        C=0.3,
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',
    )
    clf.fit(X, df[label].values)
    test_probs[:, label_idx] = clf.predict_proba(X_test)[:, 1]
    print(f'{label:20s} trained')

for i, label in enumerate(LABELS):
    df_test[f'p_{label}'] = test_probs[:, i]